# PaveX: Baseline + cumulative ablation (B0 → B3)

Answers the reviewer question *"How much better is this than GroundingDINO + SAM with no extra controls?"*

| Config | Components switched on (cumulative) |
|---|---|
| **B0** | GroundingDINO + box NMS + SAM only (no road-band, no huge-box rule, no mask NMS, no nested suppression, no post-filters) |
| **B1** | B0 + road-band filter (incl. huge-box rule) |
| **B2** | B1 + mask NMS + nested suppression |
| **B3** | B2 + post-filters = **full pipeline** |
| B3_rawmask | B3's exported instances scored on their **raw SAM masks** (measures the export loss) |

**How it works**
1. **Cache** (Step 1): GroundingDINO + SAM run **once** per image. The cache keeps one SAM mask for **every raw box** (before road-band and box NMS), so all configs share identical model outputs.
2. **Equivalence check** (Step 2): on the first 5 GT images, cached B3 must reproduce the original `predict_image()` exactly. The notebook stops if it does not.
3. **Ablation** (Step 3): one function `run_config()` applies the switched-on filters to the cache. Every config uses the same export (mask → YOLOv8-seg polygon) → rasterise (same reader as the GT) → evaluate.

**Rules:** no threshold is changed or tuned; only components are switched on/off. Run this file **exactly as committed**. Do not edit it in Colab. The only input is the git commit hash, typed into a prompt.

## Cell 1 · Environment, paths and ablation settings
Same Colab/local path logic as `fullexportrun2.ipynb`. `N_IMAGES = None` is the real run on all 200 GT images; a number is a quick test.

In [ ]:
# ============================================================
# Cell 1 — Environment, paths and ablation settings
# Works on Google Colab AND locally. Run the file AS COMMITTED: do not edit it in Colab.
# ============================================================
import sys, os, re, json, time, random, hashlib, platform, subprocess
from datetime import datetime
from pathlib import Path

NOTEBOOK_VERSION = "baseline_ablation v1.0 (2026-10-10)"

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Same packages and versions as fullexportrun2.ipynb (Colab already has torch/torchvision with CUDA)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==4.57.1", "accelerate==1.11.0", "timm==1.0.22",
                    "huggingface_hub==0.36.0", "segment-anything==1.0",
                    "opencv-python-headless==4.10.0.84", "tqdm>=4.66.0"], check=True)

# ---------------- PATHS (same as fullexportrun2.ipynb, Cell 0) ----------------
if IN_COLAB:
    REPO_ROOT  = None
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    PACK_DIR   = DRIVE_ROOT / "pavex_bbox_experiments" / "exports" / "pavex_sam_seed_pack_v11b"
    PATHS = {
        "config_dir":      PACK_DIR,
        "sam_weights":     PACK_DIR / "models" / "sam_vit_h_4b8939.pth",
        "images_dir":      DRIVE_ROOT / "BigDataset" / "augmented",
        "eval_images_dir": DRIVE_ROOT / "BigDataset" / "originals",
        "gt_labels_dir":   DRIVE_ROOT / "BigDataset" / "labels_gt",
        "runs_dir":        DRIVE_ROOT / "BigDataset" / "runs",
    }
else:
    REPO_ROOT  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    paths_file = REPO_ROOT / "configs" / "paths.local.json"
    if not paths_file.exists():
        raise FileNotFoundError(f"{paths_file} not found. Copy configs/paths.example.json to "
                                f"configs/paths.local.json and set your folders.")
    raw = json.loads(paths_file.read_text(encoding="utf-8-sig"))
    PATHS = {k: (Path(v) if Path(v).is_absolute() else (REPO_ROOT / v).resolve())
             for k, v in raw.items() if not k.startswith("_")}

CONFIG_DIR    = Path(PATHS["config_dir"])
EVAL_IMG_DIR  = Path(PATHS["eval_images_dir"])   # ORIGINAL images that the GT labels belong to
GT_LABELS_DIR = Path(PATHS["gt_labels_dir"])     # the 200 human-annotated YOLOv8-seg labels
RUNS_DIR      = Path(PATHS["runs_dir"])

# ---------------- ABLATION SETTINGS ----------------
N_IMAGES        = None    # None = all 200 GT images (the real run). A number = quick test on the first N.
BUILD_CACHE     = True    # Step 1: run GroundingDINO + SAM once per image and store the raw outputs
RUN_EQUIV_CHECK = True    # Step 2: prove the cached path reproduces the original pipeline exactly
EQUIV_N_IMAGES  = 5       # images used by the equivalence check (first 5 GT images)
RUN_ABLATION    = True    # Step 3: run B0-B3 (+ B3_rawmask) on the cache, evaluate and save
RUN_FIGURE      = False   # Step 4 (Phase 6 only): B0 vs B3 comparison figure
FIGURE_IMAGES   = []      # chosen in Phase 6, e.g. ["img_0002.jpg"]
FIGURE_RUN_DIR  = None    # results folder to draw from; None = this run's folder
SEED            = 0
EXPECTED_N_GT_IMAGES = 200

# Cumulative ablation: each config switches components ON. No threshold is ever changed.
CONFIGS = {
    "B0": {"USE_ROAD_BAND": False, "USE_MASK_NMS": False, "USE_NESTED": False, "USE_POSTFILTERS": False},
    "B1": {"USE_ROAD_BAND": True,  "USE_MASK_NMS": False, "USE_NESTED": False, "USE_POSTFILTERS": False},
    "B2": {"USE_ROAD_BAND": True,  "USE_MASK_NMS": True,  "USE_NESTED": True,  "USE_POSTFILTERS": False},
    "B3": {"USE_ROAD_BAND": True,  "USE_MASK_NMS": True,  "USE_NESTED": True,  "USE_POSTFILTERS": True},
}
RAWMASK_NAME = "B3_rawmask"   # extra row: B3's exported instances scored on raw SAM masks

# Same seed for everything (inference has no randomness and all configs read one cache; set anyway)
import numpy as np
random.seed(SEED)
np.random.seed(SEED)

IS_FULL_RUN = N_IMAGES is None
RUN_TAG   = ("full_" if IS_FULL_RUN else f"test{N_IMAGES}_") + datetime.now().strftime("%Y%m%d_%H%M%S")
ABL_ROOT  = RUNS_DIR / "baseline_ablation"
CACHE_DIR = ABL_ROOT / "cache"      # raw model outputs, shared by every config and every later run
OUT_ROOT  = ABL_ROOT / RUN_TAG      # results of THIS run: one subfolder per config
for d in [CACHE_DIR, OUT_ROOT]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------- GIT COMMIT (provenance) ----------------
def _resolve_git_commit():
    if IN_COLAB:
        # Colab cannot read the commit itself: type it in (GitHub Desktop > History > short hash)
        return input("Paste the short git commit hash of the committed baseline_ablation.ipynb: ").strip()
    try:
        c = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=REPO_ROOT,
                           capture_output=True, text=True, check=True).stdout.strip()
        dirty = subprocess.run(["git", "status", "--porcelain", "--", "notebooks/baseline_ablation.ipynb"],
                               cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
        return c + ("+notebook-uncommitted" if dirty else "")
    except Exception:
        return ""

GIT_COMMIT = _resolve_git_commit()
if IS_FULL_RUN and not re.fullmatch(r"[0-9a-f]{7,40}", GIT_COMMIT):
    raise RuntimeError(f"Full run refused: git commit hash is missing or invalid ({GIT_COMMIT!r}). "
                       "Commit baseline_ablation.ipynb in GitHub Desktop, then run again and paste the hash.")
if not GIT_COMMIT:
    GIT_COMMIT = "none (test run)"

print("Environment :", "Google Colab" if IN_COLAB else "Local")
for k, v in PATHS.items():
    print(f"{k:16s}: {v}  {'' if Path(v).exists() else '<-- NOT FOUND'}")
print("Run         :", "FULL (all 200 GT images)" if IS_FULL_RUN else f"TEST (first {N_IMAGES} GT images)")
print("Cache dir   :", CACHE_DIR)
print("Results dir :", OUT_ROOT)
print("Git commit  :", GIT_COMMIT, "| notebook:", NOTEBOOK_VERSION)

## Cell 2 · Configuration
Identical to Cell 1 of `fullexportrun2.ipynb`, so thresholds are read the same way. Cell 6 then checks every value against the frozen table.

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 1 (only change: config copies go to this run's folder, OUT_ROOT). DO NOT EDIT. <<<
# ============================================================
# Cell 1 — Read pipeline_config.json + seed_input.json
# ============================================================
import json, shutil, zipfile

# On Colab the pack may still be zipped next to its folder
PACK_ZIP = CONFIG_DIR.parent / f"{CONFIG_DIR.name}.zip"
if PACK_ZIP.exists() and not CONFIG_DIR.exists():
    print("Unzipping", PACK_ZIP.name, "...")
    with zipfile.ZipFile(PACK_ZIP) as zf:
        zf.extractall(CONFIG_DIR.parent)

PIPELINE_CFG = CONFIG_DIR / "pipeline_config.json"
SEED_INPUT   = CONFIG_DIR / "seed_input.json"

if not PIPELINE_CFG.exists():
    raise FileNotFoundError(f"pipeline_config.json not found at {PIPELINE_CFG}")

with open(PIPELINE_CFG, "r") as f:
    pipe = json.load(f)

# ---- MODELS ----
paths_info = pipe.get("paths", {})
HF_REPO    = paths_info.get("hf_repo_grounding_dino", "IDEA-Research/grounding-dino-base")
SAM_ARCH   = pipe.get("models", {}).get("SAM", {}).get("arch", None)

# ---- PROMPTS ----
# Note: the prompts already end in ".", so joining with " . " gives "potholes. . pothole. . ..."
# This is how the paper's results were produced, so it is intentionally left unchanged.
PROMPTS_MP = pipe.get("prompts", ["pothole", "road hole", "damaged road"])

# ---- THRESHOLDS ----
thr = pipe.get("thresholds", {})
TEXT_THR = float(thr.get("TEXT_THR", 0.25))   # kept for reference; does not affect box filtering
BOX_THR  = float(thr.get("BOX_THR",  0.35))
NMS_THR  = float(thr.get("NMS_THR",  0.45))

# ---- POST-FILTERS (mask area, solidity, edge rule) ----
post_filters = pipe.get("post_filters", {})
MAX_AREA_FRAC          = float(post_filters.get("max_area_frac", 0.65))
edge_cfg               = post_filters.get("edge_touches_rule", {})
EDGE_RULE_ON           = bool(edge_cfg.get("on", False))
EDGE_AREA_FRAC_GATE    = float(edge_cfg.get("area_frac_gate", 0.40))
EDGE_MIN_EDGES_TOUCHED = int(edge_cfg.get("min_edges_touched", 3))

# seed_input.json overrides min_area, max_area_frac, solidity_min
MIN_MASK_AREA = 200.0
SOLIDITY_MIN  = 0.48
if SEED_INPUT.exists():
    with open(SEED_INPUT, "r") as f:
        seed_in = json.load(f)
    pf_seed = (seed_in.get("POST_FILTERS") or {})
    MIN_MASK_AREA = float(pf_seed.get("min_area", MIN_MASK_AREA))
    MAX_AREA_FRAC = float(pf_seed.get("max_area_frac", MAX_AREA_FRAC))
    SOLIDITY_MIN  = float(pf_seed.get("solidity_min", SOLIDITY_MIN))

# ---- MASK CONSOLIDATION ----
MASK_NMS_IOU          = 0.70   # drop duplicate masks overlapping more than this
NESTED_CONTAINMENT    = 0.90   # child counts as "inside" if >= 90% of it lies in the parent
NESTED_AREA_RATIO_MAX = 0.70   # ... and the child is at most 70% of the parent's area

# ---- SAM WEIGHTS (fail fast) ----
SAM_WEIGHTS_PATH = Path(PATHS["sam_weights"])
if SAM_ARCH is None:
    raise ValueError("models.SAM.arch missing in pipeline_config.json")
if not SAM_WEIGHTS_PATH.exists():
    raise FileNotFoundError(
        f"SAM weights not found at {SAM_WEIGHTS_PATH}. Download sam_vit_h_4b8939.pth from "
        "https://github.com/facebookresearch/segment-anything and update the path."
    )

# ---- Save a copy of the config inside the run folder (provenance) ----
for p in [PIPELINE_CFG, SEED_INPUT]:
    if p.exists():
        shutil.copy2(p, OUT_ROOT / p.name)

print("HF_REPO          :", HF_REPO)
print("PROMPTS_MP       :", PROMPTS_MP)
print("BOX_THR, NMS_THR :", BOX_THR, NMS_THR)
print("SAM_ARCH         :", SAM_ARCH)
print("SAM_WEIGHTS_PATH :", SAM_WEIGHTS_PATH)
print("--- Post-filters ---")
print("MIN_MASK_AREA    :", MIN_MASK_AREA)
print("MAX_AREA_FRAC    :", MAX_AREA_FRAC)
print("SOLIDITY_MIN     :", SOLIDITY_MIN)
print("EDGE_RULE_ON     :", EDGE_RULE_ON, "| gate:", EDGE_AREA_FRAC_GATE, "| edges:", EDGE_MIN_EDGES_TOUCHED)

## Cell 3 · Load models
Identical to Cell 2 of `fullexportrun2.ipynb`, plus fixed seeds.

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 2 (loading part only). DO NOT EDIT. <<<
# ============================================================
# Cell 2 — Load GroundingDINO & SAM
# ============================================================
import platform
import torch
import transformers
from transformers import AutoProcessor, GroundingDinoForObjectDetection
from segment_anything import sam_model_registry, SamPredictor

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device, f"({torch.cuda.get_device_name(0)})" if device == "cuda" else
      "(no NVIDIA GPU: expect roughly 10-30 s per image)")

# GroundingDINO (downloaded from Hugging Face on first run, then cached)
processor   = AutoProcessor.from_pretrained(HF_REPO)
gdino_model = GroundingDinoForObjectDetection.from_pretrained(HF_REPO).to(device).eval()

# SAM
print("Loading SAM:", SAM_ARCH, "from", SAM_WEIGHTS_PATH)
sam_model = sam_model_registry[SAM_ARCH](checkpoint=str(SAM_WEIGHTS_PATH))
sam_model.to(device)
sam_predictor = SamPredictor(sam_model)
print("Models loaded.")

# ---- added for the ablation: fixed seeds, deterministic kernels ----
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

## Cells 4–5 · Original pipeline functions (verbatim)
Copied character-for-character from `fullexportrun2.ipynb` (Cells 3, 4 and the export part of Cell 5). The original `predict_image()` is kept as the **reference** for the equivalence check.

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 3. DO NOT EDIT. <<<
# ============================================================
# Cell 3 — Generic utilities: IoU, overlays, masks, GT reader
# ============================================================
import cv2
import numpy as np
from tqdm import tqdm

IMG_EXTS = [".jpg", ".jpeg", ".png", ".JPG", ".JPEG", ".PNG"]

# ------------ BOX IOU / NMS ------------
def iou_xyxy(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter_w = max(0.0, x2 - x1)
    inter_h = max(0.0, y2 - y1)
    inter = inter_w * inter_h
    if inter == 0:
        return 0.0
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    if union <= 0:
        return 0.0
    return inter / union

def nms_boxes(boxes, scores, iou_thr=0.45):
    if len(boxes) == 0:
        return boxes, scores
    boxes = boxes.copy()
    scores = scores.copy()
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(i)
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([iou_xyxy(boxes[i], boxes[j]) for j in rest])
        rest = rest[ious < iou_thr]
        order = rest
    keep = np.array(keep, dtype=int)
    return boxes[keep], scores[keep]

# ------------ LABEL PATH → IMAGE PATH (Roboflow-style) ------------
def label_to_image_path(label_path: Path, img_dir: Path):
    """
    Map Roboflow-style labels to image filenames:
      img_0002_jpg.rf.xxxxx.txt  -> <img_dir>/img_0002.jpg (or .png/.jpeg)
    """
    stem = label_path.stem
    img_stem = stem
    for marker in ["_jpg.rf.", "_png.rf.", "_jpeg.rf."]:
        if marker in stem:
            img_stem = stem.split(marker)[0]
            break

    candidates = [img_dir / f"{img_stem}{ext}" for ext in IMG_EXTS]
    for p in candidates:
        if p.exists():
            return p

    print(f"[WARN] No image found matching label {label_path.name}")
    return None

# ------------ OVERLAYS ------------
def save_overlay_boxes_and_masks(
    img_path: Path,
    boxes_xyxy,
    masks,
    out_dir: Path,
):
    """
    Draw predicted masks (transparent red) + boxes (green).
    """
    out_dir.mkdir(parents=True, exist_ok=True)
    img = cv2.imread(str(img_path))
    if img is None:
        return
    overlay = img.copy()

    # masks
    if masks is not None:
        for m in masks:
            m_bin = (m > 0).astype(np.uint8)
            if m_bin.sum() == 0:
                continue
            color = np.zeros_like(img)
            color[:, :, 2] = 255  # red
            mask_3 = m_bin[..., None]
            overlay = np.where(
                mask_3 == 1,
                (0.2 * color + 0.8 * overlay).astype(overlay.dtype),
                overlay,
            )

    # boxes
    if boxes_xyxy is not None and len(boxes_xyxy) > 0:
        for (x1, y1, x2, y2) in boxes_xyxy.astype(int):
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (0, 255, 0), 2)

    out_path = out_dir / img_path.name
    cv2.imwrite(str(out_path), overlay)

# ------------ MASK METRICS ------------
def mask_iou(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    if union == 0:
        return 0.0
    return inter / union

def mask_dice(mask1: np.ndarray, mask2: np.ndarray):
    m1 = (mask1 > 0)
    m2 = (mask2 > 0)
    inter = np.logical_and(m1, m2).sum()
    s1 = m1.sum()
    s2 = m2.sum()
    denom = s1 + s2
    if denom == 0:
        return 0.0
    return 2.0 * inter / denom

# ------------ MASK NMS ------------
# Both functions return the INDICES of the masks to keep, so the caller can keep
# masks, boxes and scores aligned (fix: previously only masks were filtered).
def mask_nms_indices(masks, scores, iou_thr=0.7):
    """Greedy mask-level NMS. Returns indices of kept masks, highest score first."""
    if len(masks) == 0:
        return []
    scores = np.asarray(scores, dtype=float)
    order = list(scores.argsort()[::-1])
    keep = []
    while order:
        i = order.pop(0)
        keep.append(int(i))
        order = [j for j in order if mask_iou(masks[i], masks[j]) <= iou_thr]
    return keep


def nested_mask_indices(masks, containment_thr=0.9, area_ratio_thr=0.7):
    """
    Drop 'child' masks that lie mostly inside a larger 'mother' mask.

    A child is dropped when:
      - containment = |child AND mother| / |child| >= containment_thr, and
      - area(child) / area(mother) <= area_ratio_thr.

    Fix: the previous version tested IoU >= 0.7 together with area ratio <= 0.7.
    Because IoU <= area ratio for any pair, both could only hold at exactly 0.7,
    so the rule almost never fired. Containment measures "inside" correctly.
    """
    n = len(masks)
    if n <= 1:
        return list(range(n))
    bins  = [(m > 0) for m in masks]
    areas = np.array([b.sum() for b in bins], dtype=float)
    keep  = np.ones(n, dtype=bool)
    for i in range(n):
        for j in range(n):
            if i == j or not keep[i] or not keep[j]:
                continue
            if areas[i] > areas[j] or (areas[i] == areas[j] and i < j):
                larger, smaller = i, j
            else:
                larger, smaller = j, i
            if areas[smaller] / max(areas[larger], 1e-6) > area_ratio_thr:
                continue
            inter = np.logical_and(bins[smaller], bins[larger]).sum()
            if inter / max(areas[smaller], 1e-6) >= containment_thr:
                keep[smaller] = False
    return [int(k) for k in np.flatnonzero(keep)]

# ------------ YOLOv8-seg GT reader ------------
def read_yolo_seg_masks(label_path: Path, img_w: int, img_h: int):
    if not label_path.exists():
        return []
    text = label_path.read_text().strip()
    if not text:
        return []

    h, w = img_h, img_w
    masks = []

    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 7:          # class id + at least 3 points
            continue
        coords = [float(v) for v in parts[1:]]
        if len(coords) % 2 != 0:
            continue
        xs = coords[0::2]
        ys = coords[1::2]
        pts = []
        for x_norm, y_norm in zip(xs, ys):
            x = int(x_norm * w)
            y = int(y_norm * h)
            pts.append([x, y])
        pts = np.array(pts, dtype=np.int32).reshape(-1, 1, 2)
        mask = np.zeros((h, w), dtype=np.uint8)
        cv2.fillPoly(mask, [pts], 255)
        masks.append(mask)

    return masks

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 4. DO NOT EDIT. <<<
# ============================================================
# Cell 4 — GDINO predictor, SAM masks, mask filters
# ============================================================
def gdino_predict_for_image(
    img_bgr: np.ndarray,
    box_thr: float = BOX_THR,
    text_thr: float = TEXT_THR,
    nms_thr: float = NMS_THR,
    margin_frac: float = 0.02,
    min_road_area_frac: float = 0.20,
    min_road_height_frac: float = 0.30,
    min_road_width_frac: float = 0.30,
):
    """
    Run GroundingDINO on one image (already loaded, BGR) and return filtered boxes.

    - Uses prompts from PROMPTS_MP.
    - Applies a simple road-band filter to remove giant bands across edges.
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]
    img_area = float(w * h)

    prompt_str = " . ".join(PROMPTS_MP)
    inputs = processor(
        images=img_rgb,
        text=prompt_str,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        threshold=float(box_thr),
        target_sizes=[(h, w)]
    )[0]

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    widths  = np.clip(x2 - x1, 0, None)
    heights = np.clip(y2 - y1, 0, None)
    areas   = widths * heights

    area_fracs   = areas / img_area
    height_fracs = heights / float(h)
    width_fracs  = widths  / float(w)

    # valid content edges (ignore small grey border)
    margin_x = int(margin_frac * w)
    margin_y = int(margin_frac * h)

    left_edge_valid   = margin_x
    right_edge_valid  = w - margin_x
    top_edge_valid    = margin_y
    bottom_edge_valid = h - margin_y

    touches_left   = x1 <= left_edge_valid
    touches_right  = x2 >= right_edge_valid
    touches_top    = y1 <= top_edge_valid
    touches_bottom = y2 >= bottom_edge_valid

    horiz_road = (
        touches_left & touches_right &
        (height_fracs >= min_road_height_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    vert_road = (
        touches_top & touches_bottom &
        (width_fracs  >= min_road_width_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    huge_box = area_fracs >= 0.75

    road_like = horiz_road | vert_road | huge_box

    keep_mask = ~road_like
    boxes  = boxes[keep_mask]
    scores = scores[keep_mask]

    if len(boxes) == 0:
        return boxes, scores, (w, h)

    boxes_nms, scores_nms = nms_boxes(boxes, scores, iou_thr=nms_thr)
    return boxes_nms, scores_nms, (w, h)


def sam_masks_from_boxes(img_bgr, boxes_xyxy, scores=None):
    """
    Run SAM for each GDINO box and return (masks, boxes, scores), index-aligned.

    Fix: if SAM returned an empty mask, the old version skipped the mask but kept
    the box, so later masks were paired with the wrong boxes.
    """
    empty = ([], np.zeros((0, 4), dtype=float), np.array([], dtype=float))
    if boxes_xyxy is None or len(boxes_xyxy) == 0:
        return empty

    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    sam_predictor.set_image(img_rgb)

    if scores is None:
        scores = np.ones(len(boxes_xyxy), dtype=float)

    masks_out, boxes_out, scores_out = [], [], []
    for box, score in zip(boxes_xyxy, scores):
        box_arr = np.array(box, dtype=np.float32)[None, :]
        with torch.no_grad():
            mask, _, _ = sam_predictor.predict(
                box=box_arr,
                point_coords=None,
                point_labels=None,
                multimask_output=False
            )
        m = mask[0]
        if m is None or m.size == 0:
            continue
        m_bin = (m > 0).astype(np.uint8)
        if m_bin.sum() == 0:
            continue
        masks_out.append(m_bin)
        boxes_out.append(np.asarray(box, dtype=float))
        scores_out.append(float(score))

    if not masks_out:
        return empty
    return masks_out, np.vstack(boxes_out), np.array(scores_out, dtype=float)


def mask_solidity(mask: np.ndarray) -> float:
    """
    Approximate solidity = area / convex_hull_area.
    """
    m = (mask > 0).astype(np.uint8)
    if m.sum() < 5:
        return 0.0
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return 0.0
    cnt = max(contours, key=cv2.contourArea)
    area = cv2.contourArea(cnt)
    if area <= 0:
        return 0.0
    hull = cv2.convexHull(cnt)
    hull_area = cv2.contourArea(hull)
    if hull_area <= 0:
        return 0.0
    return float(area / hull_area)


def filter_masks_with_postfilters(masks, boxes_xyxy, scores,
                                  min_area: float = MIN_MASK_AREA,
                                  max_area_frac: float = MAX_AREA_FRAC,
                                  solidity_min: float = SOLIDITY_MIN,
                                  edge_rule_on: bool = EDGE_RULE_ON,
                                  edge_area_frac_gate: float = EDGE_AREA_FRAC_GATE,
                                  edge_min_edges_touched: int = EDGE_MIN_EDGES_TOUCHED):
    """
    Apply area / solidity / edge-touch filters using config from pack.
    Returns filtered (masks, boxes, scores).
    """
    if len(masks) == 0:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    h, w = masks[0].shape[:2]
    img_area = float(w * h)

    kept_masks = []
    kept_boxes = []
    kept_scores = []

    for m, box, s in zip(masks, boxes_xyxy, scores):
        area = (m > 0).sum()
        if area < min_area:
            continue

        area_frac = area / img_area
        if area_frac > max_area_frac:
            continue

        sol = mask_solidity(m)
        if sol < solidity_min:
            continue

        # optional edge-touch rule (road-like giant region)
        if edge_rule_on:
            x1, y1, x2, y2 = box
            touches_top = y1 <= 0.01 * h
            touches_bottom = y2 >= 0.99 * h
            touches_left = x1 <= 0.01 * w
            touches_right = x2 >= 0.99 * w
            edges_touched = sum([touches_top, touches_bottom, touches_left, touches_right])
            if area_frac >= edge_area_frac_gate and edges_touched >= edge_min_edges_touched:
                # likely huge border region
                continue

        kept_masks.append(m)
        kept_boxes.append(box)
        kept_scores.append(s)

    if not kept_masks:
        return [], np.zeros((0, 4), dtype=float), np.array([], dtype=float)

    return kept_masks, np.vstack(kept_boxes), np.array(kept_scores, dtype=float)


def predict_image(img_bgr):
    """
    Full pipeline for ONE image, used by both auto-annotation (Cell 5) and
    evaluation (Cell 6) so they can never drift apart.

    Returns (masks, boxes, scores); element k of each belongs to the same detection.
    """
    # 1) GroundingDINO boxes + road-band suppression + box NMS
    boxes, scores, _ = gdino_predict_for_image(img_bgr)

    # 2) SAM box-prompted masks (aligned with boxes)
    masks, boxes, scores = sam_masks_from_boxes(img_bgr, boxes, scores)

    # 3) Mask NMS, then nested-mask suppression; index-based so boxes stay aligned
    keep = mask_nms_indices(masks, scores, iou_thr=MASK_NMS_IOU)
    masks, boxes, scores = [masks[k] for k in keep], boxes[keep], scores[keep]

    keep = nested_mask_indices(masks, containment_thr=NESTED_CONTAINMENT,
                               area_ratio_thr=NESTED_AREA_RATIO_MAX)
    masks, boxes, scores = [masks[k] for k in keep], boxes[keep], scores[keep]

    # 4) Post-filters (area, solidity, edge rule)
    return filter_masks_with_postfilters(masks, boxes, scores)

In [ ]:
# >>> COPIED VERBATIM from notebooks/fullexportrun2.ipynb, Cell 5 (functions only; auto_annotate_all_images removed). DO NOT EDIT. <<<
# ============================================================
# Cell 5 — Binary mask → YOLOv8-seg label + auto-annotate all
# ============================================================
import os

def binary_mask_to_yolov8_seg_line(
    mask,
    img_w: int,
    img_h: int,
    cls_id: int = 0,
    simplify_eps: float = 1.5,
):
    """
    Convert a binary mask (H,W) into one YOLOv8-seg label line:
      cls x1 y1 x2 y2 ... xn yn      (normalised to [0, 1])

    Fix: the old version also wrote "cx cy w h" after the class id. YOLOv8-seg
    reads every number after the class id as polygon points, so those four values
    became two fake vertices and corrupted every mask.
    """
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return None

    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    cnt = max(contours, key=cv2.contourArea)       # largest connected region only
    if cv2.contourArea(cnt) < 10:
        return None

    if simplify_eps > 0:
        approx = cv2.approxPolyDP(cnt, simplify_eps, True)
        if len(approx) >= 3:
            cnt = approx

    pts = cnt.reshape(-1, 2).astype(float)
    if len(pts) < 3:
        return None

    xs = np.clip(pts[:, 0] / img_w, 0.0, 1.0)
    ys = np.clip(pts[:, 1] / img_h, 0.0, 1.0)

    parts = [str(cls_id)]
    for x, y in zip(xs, ys):
        parts += [f"{x:.6f}", f"{y:.6f}"]
    return " ".join(parts)


def _write_atomic(path: Path, text: str):
    """Write to a temp file, then rename. An interrupted run never leaves a half-written label."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text)
    os.replace(tmp, path)


def list_images(folder: Path):
    # Deduplicate: Windows file systems are case-insensitive, so *.jpg and *.JPG
    # would otherwise return the same files twice.
    found = {p.resolve() for ext in IMG_EXTS for p in folder.glob(f"*{ext}")}
    return sorted(found)

## Cell 6 · Frozen-config check and config snapshot
Stops if any threshold differs from the frozen table. It also checks the values that are hard-coded inside the original functions.

In [ ]:
# ============================================================
# Cell 6 — Check the frozen configuration (stops if anything differs)
# ============================================================
import inspect

FROZEN = {
    "PROMPTS": ["potholes.", "pothole.", "road pothole.", "asphalt hole."],
    "HF_REPO": "IDEA-Research/grounding-dino-base",
    "BOX_THR": 0.24, "NMS_THR": 0.33,
    "MARGIN_FRAC": 0.02, "MIN_ROAD_AREA_FRAC": 0.20, "MIN_ROAD_HEIGHT_FRAC": 0.30,
    "MIN_ROAD_WIDTH_FRAC": 0.30, "HUGE_BOX_AREA_FRAC": 0.75,
    "SAM_ARCH": "vit_h", "MULTIMASK_OUTPUT": False,
    "MASK_NMS_IOU": 0.70, "NESTED_CONTAINMENT": 0.90, "NESTED_AREA_RATIO_MAX": 0.70,
    "MIN_MASK_AREA": 200.0, "MAX_AREA_FRAC": 0.50, "SOLIDITY_MIN": 0.60,
    "EDGE_RULE_ON": True, "EDGE_AREA_FRAC_GATE": 0.40, "EDGE_MIN_EDGES_TOUCHED": 3,
    "MIN_CONTOUR_AREA_PX": 10, "SIMPLIFY_EPS_PX": 1.5,
    "IOU_THR_EVAL": 0.5,
}

_gd = inspect.signature(gdino_predict_for_image).parameters
LOADED = {
    "PROMPTS": list(PROMPTS_MP), "HF_REPO": HF_REPO,
    "BOX_THR": BOX_THR, "NMS_THR": NMS_THR,
    "MARGIN_FRAC": _gd["margin_frac"].default,
    "MIN_ROAD_AREA_FRAC": _gd["min_road_area_frac"].default,
    "MIN_ROAD_HEIGHT_FRAC": _gd["min_road_height_frac"].default,
    "MIN_ROAD_WIDTH_FRAC": _gd["min_road_width_frac"].default,
    "SAM_ARCH": SAM_ARCH,
    "MASK_NMS_IOU": MASK_NMS_IOU, "NESTED_CONTAINMENT": NESTED_CONTAINMENT,
    "NESTED_AREA_RATIO_MAX": NESTED_AREA_RATIO_MAX,
    "MIN_MASK_AREA": MIN_MASK_AREA, "MAX_AREA_FRAC": MAX_AREA_FRAC, "SOLIDITY_MIN": SOLIDITY_MIN,
    "EDGE_RULE_ON": EDGE_RULE_ON, "EDGE_AREA_FRAC_GATE": EDGE_AREA_FRAC_GATE,
    "EDGE_MIN_EDGES_TOUCHED": EDGE_MIN_EDGES_TOUCHED,
    "IOU_THR_EVAL": 0.5,
}
# Values written as literals inside the original functions: check the source text
_src_gd  = inspect.getsource(gdino_predict_for_image)
_src_sam = inspect.getsource(sam_masks_from_boxes)
_src_exp = inspect.getsource(binary_mask_to_yolov8_seg_line)
_src_pi  = inspect.getsource(predict_image)
LOADED["HUGE_BOX_AREA_FRAC"]  = 0.75 if "huge_box = area_fracs >= 0.75" in _src_gd else "NOT FOUND"
LOADED["MULTIMASK_OUTPUT"]    = False if "multimask_output=False" in _src_sam else "NOT FOUND"
LOADED["MIN_CONTOUR_AREA_PX"] = 10 if "cv2.contourArea(cnt) < 10" in _src_exp else "NOT FOUND"
LOADED["SIMPLIFY_EPS_PX"]     = inspect.signature(binary_mask_to_yolov8_seg_line).parameters["simplify_eps"].default

mismatch = {k: {"frozen": v, "found": LOADED.get(k)} for k, v in FROZEN.items() if LOADED.get(k) != v}
if mismatch:
    raise ValueError(f"Configuration differs from the frozen table, stopping: {mismatch}")
print("Frozen configuration verified:", len(FROZEN), "values match.")


def sha256_json(obj):
    return hashlib.sha256(json.dumps(obj, sort_keys=True).encode("utf-8")).hexdigest()


def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def config_snapshot(name):
    """Everything that defines a config: its switches plus every frozen threshold and setting."""
    flags = CONFIGS["B3"] if name == RAWMASK_NAME else CONFIGS[name]
    return {
        "config": name,
        "flags": dict(flags),
        "prediction_masks": ("raw SAM masks of the instances that survive B3 export"
                             if name == RAWMASK_NAME else
                             "exported YOLOv8-seg polygons, rasterised with read_yolo_seg_masks (same reader as GT)"),
        "frozen": FROZEN,
        "prompt_string": " . ".join(PROMPTS_MP),
        "text_threshold_note": "TEXT_THR is not passed to post-processing; boxes are filtered by BOX_THR only",
        "pipeline_order": "[road-band] -> box NMS -> drop empty SAM masks -> [mask NMS] -> [nested] -> [post-filters]",
        "export": "largest external contour only, approxPolyDP eps 1.5 px, contour area >= 10 px, coords clipped to [0,1]",
        "evaluation": {
            "subset": "200-image human-annotated GT subset (labels_gt)",
            "gt": "YOLOv8-seg polygons, one polygon = one instance, filled to binary masks",
            "matching": "greedy one-to-one by decreasing mask IoU; TP if IoU >= 0.5",
            "averaging": "micro over all images; mean IoU/Dice over TP pairs only",
            "export_drop_rule": "masks with no exported polygon are not predictions (not FP); unmatched GT stay FN",
        },
        "seed": SEED,
    }

## Cell 7 · Helpers for the cached path
Each helper is **one piece of an original function**, split so the cache can sit in between. The equivalence check (Step 2) proves the split changes nothing.

In [ ]:
# ============================================================
# Cell 7 — Helpers for the cached path (pieces of the original functions)
# ============================================================
def gdino_raw_boxes(img_bgr, box_thr: float = BOX_THR):
    """First half of gdino_predict_for_image: the model call only (NO road-band, NO box NMS)."""
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w = img_rgb.shape[:2]

    prompt_str = " . ".join(PROMPTS_MP)
    inputs = processor(
        images=img_rgb,
        text=prompt_str,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = gdino_model(**inputs)

    results = processor.post_process_grounded_object_detection(
        outputs,
        threshold=float(box_thr),
        target_sizes=[(h, w)]
    )[0]

    boxes  = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    return boxes, scores


def road_band_keep(boxes, w, h,
                   margin_frac: float = 0.02,
                   min_road_area_frac: float = 0.20,
                   min_road_height_frac: float = 0.30,
                   min_road_width_frac: float = 0.30):
    """Second half of gdino_predict_for_image: the road-band + huge-box rule. Returns True = keep.
    Uses only box coordinates and image size (never SAM output)."""
    if len(boxes) == 0:
        return np.zeros(0, dtype=bool)
    img_area = float(w * h)

    x1 = boxes[:, 0]
    y1 = boxes[:, 1]
    x2 = boxes[:, 2]
    y2 = boxes[:, 3]

    widths  = np.clip(x2 - x1, 0, None)
    heights = np.clip(y2 - y1, 0, None)
    areas   = widths * heights

    area_fracs   = areas / img_area
    height_fracs = heights / float(h)
    width_fracs  = widths  / float(w)

    margin_x = int(margin_frac * w)
    margin_y = int(margin_frac * h)

    left_edge_valid   = margin_x
    right_edge_valid  = w - margin_x
    top_edge_valid    = margin_y
    bottom_edge_valid = h - margin_y

    touches_left   = x1 <= left_edge_valid
    touches_right  = x2 >= right_edge_valid
    touches_top    = y1 <= top_edge_valid
    touches_bottom = y2 >= bottom_edge_valid

    horiz_road = (
        touches_left & touches_right &
        (height_fracs >= min_road_height_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    vert_road = (
        touches_top & touches_bottom &
        (width_fracs  >= min_road_width_frac) &
        (area_fracs   >= min_road_area_frac)
    )
    huge_box = area_fracs >= 0.75

    road_like = horiz_road | vert_road | huge_box
    return ~road_like


def nms_box_indices(boxes, scores, iou_thr=NMS_THR):
    """Same loop as nms_boxes, but returns the kept INDICES (in kept order, highest score first)."""
    if len(boxes) == 0:
        return []
    order = scores.argsort()[::-1]
    keep = []
    while len(order) > 0:
        i = order[0]
        keep.append(int(i))
        if len(order) == 1:
            break
        rest = order[1:]
        ious = np.array([iou_xyxy(boxes[i], boxes[j]) for j in rest])
        rest = rest[ious < iou_thr]
        order = rest
    return keep


def sam_mask_for_box(box):
    """One SAM call for one box, exactly as inside sam_masks_from_boxes (set_image must be done first).
    Returns a boolean mask; all-False if SAM returned nothing."""
    box_arr = np.array(box, dtype=np.float32)[None, :]
    with torch.no_grad():
        mask, _, _ = sam_predictor.predict(
            box=box_arr,
            point_coords=None,
            point_labels=None,
            multimask_output=False
        )
    m = mask[0]
    if m is None or m.size == 0:
        return np.zeros(sam_predictor.original_size, dtype=bool)
    return m > 0


# The helpers must use the same defaults as the original function
_rb = inspect.signature(road_band_keep).parameters
for _k in ["margin_frac", "min_road_area_frac", "min_road_height_frac", "min_road_width_frac"]:
    assert _rb[_k].default == _gd[_k].default, f"road_band_keep default {_k} differs from the original"
print("Cached-path helpers defined.")

## Cell 8 · Synthetic test of the nested-suppression rule (hand-made masks, no data)
Shows that the containment rule (containment ≥ 0.90 and area ratio ≤ 0.70) behaves as designed. It also shows why the old condition (IoU ≥ 0.70 and area ratio ≤ 0.70) almost never fired.

In [ ]:
# ============================================================
# Cell 8 — Synthetic nested-suppression test (no images, no GT)
# ============================================================
def _square(x, y, s, canvas=300):
    m = np.zeros((canvas, canvas), dtype=np.uint8)
    m[y:y + s, x:x + s] = 1
    return m

PARENT = _square(50, 50, 100)                                    # 100 x 100 = 10,000 px
SYNTH_CASES = [
    # (description, child mask, child expected to be dropped?)
    ("child fully inside, area ratio 0.09", _square(60, 60, 30),  True),
    ("child fully inside, area ratio 0.56", _square(60, 60, 75),  True),
    ("child fully inside, area ratio 0.81", _square(55, 55, 90),  False),  # ratio > 0.70: kept
    ("separate mask (no overlap)",          _square(200, 200, 30), False),
    ("partly overlapping (50% inside)",     _square(130, 60, 40), False),  # containment 0.5 < 0.90
]

print(f"{'case':40s} {'contain':>8s} {'ratio':>6s} {'IoU':>6s}  new rule   old IoU rule")
for desc, child, expect_drop in SYNTH_CASES:
    keep = nested_mask_indices([PARENT, child], containment_thr=NESTED_CONTAINMENT,
                               area_ratio_thr=NESTED_AREA_RATIO_MAX)
    dropped = 1 not in keep
    inter = np.logical_and(PARENT > 0, child > 0).sum()
    contain = inter / child.sum()
    ratio = child.sum() / PARENT.sum()
    iou = mask_iou(PARENT, child)
    old_drop = (iou >= 0.70) and (ratio <= 0.70)       # the pre-fix condition, for comparison only
    print(f"{desc:40s} {contain:8.2f} {ratio:6.2f} {iou:6.2f}  "
          f"{'DROP' if dropped else 'keep':9s}  {'DROP' if old_drop else 'keep'}")
    assert dropped == expect_drop, f"nested rule failed on: {desc}"
    assert 0 in keep, "the parent must always be kept"
print("Synthetic nested-suppression test passed.")

## Cell 9 · Step 1: build the cache (GroundingDINO + SAM, once per image)
For every GT image: all raw GroundingDINO boxes and scores, plus **one SAM mask per raw box**. SAM computes the image embedding once, then prompts each box separately. One compressed file per image. Re-running skips images that are already cached.

In [ ]:
# ============================================================
# Cell 9 — Step 1: cache raw model outputs
# ============================================================
CACHE_META = {
    "hf_repo": HF_REPO, "sam_arch": SAM_ARCH, "sam_weights": SAM_WEIGHTS_PATH.name,
    "prompts": list(PROMPTS_MP), "box_thr": BOX_THR, "multimask_output": False,
    "stored": "all raw GDINO boxes (before road-band and box NMS) + one SAM mask per box",
}


def all_gt_label_paths():
    """The 200-image human-annotated subset. Always checks the folder really holds 200 labels."""
    paths = sorted(GT_LABELS_DIR.glob("*.txt"))
    if len(paths) != EXPECTED_N_GT_IMAGES:
        raise RuntimeError(f"Expected {EXPECTED_N_GT_IMAGES} GT labels in {GT_LABELS_DIR}, found {len(paths)}. "
                           "This must be the 200-image subset (never the 50-image diagnostic subset).")
    return paths


def gt_label_paths():
    paths = all_gt_label_paths()
    return paths if IS_FULL_RUN else paths[:N_IMAGES]


def cache_file(lbl_path):
    return CACHE_DIR / f"{lbl_path.stem}.npz"


def build_cache_entry(lbl_path):
    img_path = label_to_image_path(lbl_path, EVAL_IMG_DIR)
    if img_path is None:
        raise FileNotFoundError(f"No original image for GT label {lbl_path.name}")
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        raise IOError(f"Cannot read {img_path}")
    h, w = img_bgr.shape[:2]

    boxes, scores = gdino_raw_boxes(img_bgr)
    n = len(boxes)
    masks = np.zeros((n, h, w), dtype=bool)
    if n > 0:
        sam_predictor.set_image(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))   # embedding once per image
        for k in range(n):
            masks[k] = sam_mask_for_box(boxes[k])                           # each box prompted on its own

    out = cache_file(lbl_path)
    tmp = out.with_name(out.stem + ".tmp.npz")
    np.savez_compressed(tmp, boxes=boxes, scores=scores,
                        masks_packed=np.packbits(masks.reshape(n, h * w), axis=1),
                        h=h, w=w, image_name=img_path.name,
                        meta=json.dumps(CACHE_META, sort_keys=True))
    os.replace(tmp, out)   # an interrupted run never leaves a half-written cache file


def load_cache_entry(lbl_path):
    f = cache_file(lbl_path)
    if not f.exists():
        raise FileNotFoundError(f"Cache missing for {lbl_path.name}; run Step 1 first.")
    with np.load(f, allow_pickle=False) as z:
        if json.loads(str(z["meta"])) != json.loads(json.dumps(CACHE_META, sort_keys=True)):
            raise RuntimeError(f"Cache file {f.name} was built with different settings; delete the cache folder.")
        h, w = int(z["h"]), int(z["w"])
        boxes, scores = z["boxes"], z["scores"]
        n = len(boxes)
        masks = np.unpackbits(z["masks_packed"], axis=1, count=h * w).reshape(n, h, w).astype(np.uint8)
        image_name = str(z["image_name"])
    return {"label_path": lbl_path, "image_name": image_name, "image_path": EVAL_IMG_DIR / image_name,
            "h": h, "w": w, "boxes": boxes, "scores": scores, "masks": masks}


if BUILD_CACHE:
    _todo = [p for p in gt_label_paths() if not cache_file(p).exists()]
    print(f"Cache: {len(gt_label_paths()) - len(_todo)} already done, {len(_todo)} to build.")
    _t0 = time.time()
    for _p in tqdm(_todo, desc="Building cache"):
        build_cache_entry(_p)
    if _todo:
        print(f"Cache built in {time.time() - _t0:.0f} s ({(time.time() - _t0) / len(_todo):.1f} s/image).")
else:
    print("BUILD_CACHE is False; using the existing cache.")

## Cell 10 · `run_config()`: one function for every config
Works on **one list of instances**. Each instance carries its `instance_id`, the index of its raw GroundingDINO box, from start to finish, so there are no parallel lists that can drift apart. After every step it checks that ids are unique, that no new id appeared, and that every instance still has its box, mask and score.

In [ ]:
# ============================================================
# Cell 10 — One function runs every config on the cache
# ============================================================
STAGES = ["raw", "road_band", "box_nms", "drop_empty", "mask_nms", "nested", "postfilters"]


def _check_instances(prev, cur, stage, h, w):
    ids = [d["id"] for d in cur]
    assert len(ids) == len(set(ids)), f"{stage}: duplicate instance ids"
    assert set(ids) <= {d["id"] for d in prev}, f"{stage}: an unknown instance id appeared"
    for d in cur:
        assert np.shape(d["box"]) == (4,), f"{stage}: bad box for instance {d['id']}"
        assert d["mask"].shape == (h, w), f"{stage}: bad mask for instance {d['id']}"
        assert np.isfinite(d["score"]), f"{stage}: bad score for instance {d['id']}"


def run_config(entry, flags):
    """
    Same order as predict_image():
      [road-band] -> box NMS -> drop empty SAM masks -> [mask NMS] -> [nested] -> [post-filters]
    [ ] = switched by flags. Returns (final instances, {stage: ids kept after that stage}).
    """
    h, w = entry["h"], entry["w"]
    inst = [{"id": k, "box": entry["boxes"][k], "score": entry["scores"][k], "mask": entry["masks"][k]}
            for k in range(len(entry["boxes"]))]
    trace = {"raw": [d["id"] for d in inst]}

    def advance(stage, new):
        nonlocal inst
        _check_instances(inst, new, stage, h, w)
        inst = new
        trace[stage] = [d["id"] for d in inst]

    # 1) Road-band filter (box geometry only)
    if flags["USE_ROAD_BAND"] and inst:
        keep = road_band_keep(np.stack([d["box"] for d in inst]), w, h)
        advance("road_band", [d for d, k in zip(inst, keep) if k])
    else:
        advance("road_band", list(inst))

    # 2) Box NMS (always on). Live cross-check against the original nms_boxes on every call.
    if inst:
        b = np.stack([d["box"] for d in inst])
        s = np.array([d["score"] for d in inst], dtype=entry["scores"].dtype)
        keep = nms_box_indices(b, s, iou_thr=NMS_THR)
        b_ref, s_ref = nms_boxes(b, s, iou_thr=NMS_THR)
        assert np.array_equal(b_ref, b[keep]) and np.array_equal(s_ref, s[keep]), "box NMS mismatch"
        advance("box_nms", [inst[k] for k in keep])
    else:
        advance("box_nms", [])

    # 3) Drop boxes whose SAM mask is empty, and convert types exactly as sam_masks_from_boxes does
    advance("drop_empty", [{"id": d["id"], "box": np.asarray(d["box"], dtype=float),
                            "score": float(d["score"]), "mask": d["mask"]}
                           for d in inst if d["mask"].sum() > 0])

    # 4) Mask NMS
    if flags["USE_MASK_NMS"]:
        keep = mask_nms_indices([d["mask"] for d in inst], [d["score"] for d in inst], iou_thr=MASK_NMS_IOU)
        advance("mask_nms", [inst[k] for k in keep])
    else:
        advance("mask_nms", list(inst))

    # 5) Nested-mask suppression (containment rule)
    if flags["USE_NESTED"]:
        keep = nested_mask_indices([d["mask"] for d in inst], containment_thr=NESTED_CONTAINMENT,
                                   area_ratio_thr=NESTED_AREA_RATIO_MAX)
        advance("nested", [inst[k] for k in keep])
    else:
        advance("nested", list(inst))

    # 6) Post-filters: the ORIGINAL function, called one instance at a time (its rules are per mask)
    if flags["USE_POSTFILTERS"]:
        advance("postfilters", [d for d in inst
                                if len(filter_masks_with_postfilters([d["mask"]], d["box"][None, :],
                                                                     np.array([d["score"]]))[0]) == 1])
    else:
        advance("postfilters", list(inst))

    return inst, trace

## Cell 11 · Export → rasterise → evaluate (identical for every config)
- **Export:** the original `binary_mask_to_yolov8_seg_line`, which keeps the largest outer contour, simplifies it with ε 1.5 px, and requires a contour of at least 10 px. One mask gives at most one line.
- **Rasterise:** the label file is read back with `read_yolo_seg_masks`, the same reader used for the GT.
- **Export drops:** a mask that yields no polygon is **not** a prediction, so it is not an FP. It is logged in `export_drops.csv`, and any GT it would have hit stays an FN.
- **Matching:** greedy one-to-one by decreasing IoU, TP if IoU ≥ 0.5. This is the same loop as Cell 6 of `fullexportrun2.ipynb`.

In [ ]:
# ============================================================
# Cell 11 — Export, rasterise, evaluate
# ============================================================
IOU_THR_EVAL = 0.5


def _export_drop_reason(mask):
    """Explains why binary_mask_to_yolov8_seg_line returned None (same checks, same order)."""
    m = (mask > 0).astype(np.uint8)
    if m.sum() == 0:
        return "empty mask"
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return "no contour"
    if cv2.contourArea(max(contours, key=cv2.contourArea)) < 10:
        return "largest contour < 10 px (MIN_CONTOUR_AREA_PX)"
    return "fewer than 3 polygon points"


def export_instances(inst, h, w, label_path):
    """Original export -> label file -> read back with the GT reader.
    Returns (preds [(instance_id, rasterised mask)], exported instances, drops, export stats)."""
    lines, kept, drops, stats = [], [], [], []
    for d in inst:
        line = binary_mask_to_yolov8_seg_line(d["mask"], img_w=w, img_h=h, cls_id=0, simplify_eps=1.5)
        if line is None:
            drops.append({"pred_id": d["id"], "raw_area_px": int((d["mask"] > 0).sum()),
                          "reason": _export_drop_reason(d["mask"])})
            continue
        lines.append(line)
        kept.append(d)
    label_path.parent.mkdir(parents=True, exist_ok=True)
    _write_atomic(label_path, "\n".join(lines) + "\n" if lines else "")

    raster = read_yolo_seg_masks(label_path, w, h)
    if len(raster) != len(kept):
        raise RuntimeError(f"{label_path.name}: wrote {len(kept)} polygons but read back {len(raster)}")
    preds = []
    for d, r in zip(kept, raster):
        n_outer = len(cv2.findContours((d["mask"] > 0).astype(np.uint8), cv2.RETR_EXTERNAL,
                                       cv2.CHAIN_APPROX_SIMPLE)[0])
        raw_area, exp_area = int((d["mask"] > 0).sum()), int((r > 0).sum())
        stats.append({"pred_id": d["id"], "n_outer_contours": n_outer, "raw_area_px": raw_area,
                      "exported_area_px": exp_area,
                      "iou_raw_vs_exported": round(float(mask_iou(d["mask"], r)), 6)})
        preds.append((d["id"], r))
    return preds, kept, drops, stats


def evaluate_image(image_name, preds, gts):
    """preds/gts: lists of (id, mask). Greedy one-to-one matching, exactly as fullexportrun2.ipynb Cell 6."""
    n_gt, n_pred = len(gts), len(preds)
    iou_mat = np.zeros((n_gt, n_pred), dtype=float)
    dice_mat = np.zeros((n_gt, n_pred), dtype=float)
    for i in range(n_gt):
        for j in range(n_pred):
            iou_mat[i, j] = mask_iou(gts[i][1], preds[j][1])
            dice_mat[i, j] = mask_dice(gts[i][1], preds[j][1])

    pairs = []
    for i in range(n_gt):
        for j in range(n_pred):
            pairs.append((iou_mat[i, j], i, j))
    pairs.sort(reverse=True, key=lambda x: x[0])

    matched_gt, matched_pred, tp_pairs = set(), set(), []
    for iou_val, i, j in pairs:
        if iou_val < IOU_THR_EVAL:
            break
        if i in matched_gt or j in matched_pred:
            continue
        matched_gt.add(i)
        matched_pred.add(j)
        tp_pairs.append((i, j))

    area = lambda m: int((m > 0).sum())
    rows = []
    for i, j in tp_pairs:
        rows.append({"image": image_name, "pred_id": preds[j][0], "gt_id": gts[i][0], "outcome": "TP",
                     "iou": round(iou_mat[i, j], 6), "dice": round(dice_mat[i, j], 6), "best_iou_any": "",
                     "pred_area_px": area(preds[j][1]), "gt_area_px": area(gts[i][1])})
    for j in range(n_pred):
        if j not in matched_pred:
            best = float(iou_mat[:, j].max()) if n_gt else 0.0
            rows.append({"image": image_name, "pred_id": preds[j][0], "gt_id": "", "outcome": "FP",
                         "iou": "", "dice": "", "best_iou_any": round(best, 6),
                         "pred_area_px": area(preds[j][1]), "gt_area_px": ""})
    for i in range(n_gt):
        if i not in matched_gt:
            best = float(iou_mat[i, :].max()) if n_pred else 0.0
            rows.append({"image": image_name, "pred_id": "", "gt_id": gts[i][0], "outcome": "FN",
                         "iou": "", "dice": "", "best_iou_any": round(best, 6),
                         "pred_area_px": "", "gt_area_px": area(gts[i][1])})
    return {"TP": len(tp_pairs), "FP": n_pred - len(tp_pairs), "FN": n_gt - len(tp_pairs),
            "ious": [float(iou_mat[i, j]) for i, j in tp_pairs],
            "dices": [float(dice_mat[i, j]) for i, j in tp_pairs],
            "n_pred_empty": sum(1 for _, m in preds if area(m) == 0),
            "rows": rows}


def summarise(TP, FP, FN, ious, dices):
    """Micro-averaged metrics; every division guarded (None = undefined, written as null)."""
    P = TP / (TP + FP) if (TP + FP) > 0 else None
    R = TP / (TP + FN) if (TP + FN) > 0 else None
    if P is None or R is None:
        F1 = None
    else:
        F1 = 2 * P * R / (P + R) if (P + R) > 0 else 0.0
    return {"TP": TP, "FP": FP, "FN": FN, "precision": P, "recall": R, "f1": F1,
            "iou_mean": float(np.mean(ious)) if ious else None,
            "dice_mean": float(np.mean(dices)) if dices else None,
            "n_tp_pairs": len(ious)}

## Cell 12 · Step 2: equivalence check (cached path vs original pipeline)
On the first 5 GT images it compares:
1. boxes after road-band and box NMS (original `gdino_predict_for_image`) with the cached path;
2. the final masks (original `predict_image`) with cached **B3**: same count, mask IoU ≈ 1.0;
3. the exported label lines.

If anything differs, the notebook **stops** before the ablation runs.

In [ ]:
# ============================================================
# Cell 12 — Step 2: equivalence check
# ============================================================
def equivalence_check(n_images):
    report = []
    for lbl in tqdm(all_gt_label_paths()[:n_images], desc="Equivalence check"):
        if not cache_file(lbl).exists():
            build_cache_entry(lbl)
        entry = load_cache_entry(lbl)
        h, w = entry["h"], entry["w"]
        img_bgr = cv2.imread(str(entry["image_path"]))
        final, trace = run_config(entry, CONFIGS["B3"])

        # (1) boxes after road-band + box NMS
        b_orig, s_orig, _ = gdino_predict_for_image(img_bgr)
        b_cache = entry["boxes"][trace["box_nms"]]
        s_cache = entry["scores"][trace["box_nms"]]
        boxes_ok = (b_orig.shape == b_cache.shape and np.allclose(b_orig, b_cache, atol=1e-3)
                    and np.allclose(s_orig, s_cache, atol=1e-4))

        # (2) final masks: original predict_image vs cached B3 (compared in order)
        m_orig, _, _ = predict_image(img_bgr)
        same_count = len(m_orig) == len(final)
        ious = [float(mask_iou(a, d["mask"])) for a, d in zip(m_orig, final)] if same_count else []
        min_iou = min(ious) if ious else (1.0 if same_count else 0.0)
        n_identical = sum(np.array_equal(a > 0, d["mask"] > 0) for a, d in zip(m_orig, final))

        # (3) exported label lines
        to_lines = lambda ms: [l for l in (binary_mask_to_yolov8_seg_line(m, img_w=w, img_h=h, cls_id=0,
                                                                          simplify_eps=1.5) for m in ms) if l]
        lines_identical = to_lines(m_orig) == to_lines([d["mask"] for d in final])

        passed = bool(boxes_ok and same_count and min_iou >= 0.99)
        report.append({"image": entry["image_name"], "passed": passed,
                       "boxes_after_nms_match": bool(boxes_ok),
                       "n_boxes_original": int(len(b_orig)), "n_boxes_cached": int(len(b_cache)),
                       "n_instances_original": len(m_orig), "n_instances_cached": len(final),
                       "min_mask_iou": round(min_iou, 6), "n_masks_pixel_identical": int(n_identical),
                       "label_lines_identical": bool(lines_identical)})
        print(report[-1])

    out = {"n_images": len(report), "all_passed": all(r["passed"] for r in report),
           "criteria": "boxes after road-band+NMS match; same instance count; every mask IoU >= 0.99",
           "git_commit": GIT_COMMIT, "notebook_version": NOTEBOOK_VERSION, "device": device,
           "images": report}
    (OUT_ROOT / "equivalence_check.json").write_text(json.dumps(out, indent=2))
    if not out["all_passed"]:
        raise RuntimeError("Equivalence check FAILED. Do not run the ablation. "
                           f"Send {OUT_ROOT / 'equivalence_check.json'} for diagnosis.")
    print(f"Equivalence check PASSED on {len(report)} images -> {OUT_ROOT / 'equivalence_check.json'}")


if RUN_EQUIV_CHECK:
    equivalence_check(EQUIV_N_IMAGES)
else:
    print("RUN_EQUIV_CHECK is False; skipping.")

## Cell 13 · Step 3: run B0–B3 (+ B3_rawmask), evaluate and save
Writes one folder per config: `metrics.json`, `config_snapshot.json`, `per_instance.csv`, `per_image.csv`, `export_drops.csv`, `export_stats.csv` and `labels/`. Plus one `summary.csv` for all configs. Uses Python's built-in `csv` module only (no pandas).

In [ ]:
# ============================================================
# Cell 13 — Step 3: ablation run + save
# ============================================================
import csv

OUTPUT_NAMES = list(CONFIGS) + [RAWMASK_NAME]
PER_INSTANCE_FIELDS = ["image", "pred_id", "gt_id", "outcome", "iou", "dice", "best_iou_any",
                       "pred_area_px", "gt_area_px"]
CODE_FUNCS = [iou_xyxy, nms_boxes, label_to_image_path, mask_iou, mask_dice, mask_nms_indices,
              nested_mask_indices, read_yolo_seg_masks, gdino_predict_for_image, sam_masks_from_boxes,
              mask_solidity, filter_masks_with_postfilters, predict_image, binary_mask_to_yolov8_seg_line,
              gdino_raw_boxes, road_band_keep, nms_box_indices, sam_mask_for_box, build_cache_entry,
              load_cache_entry, run_config, export_instances, evaluate_image, summarise, config_snapshot]


def code_hashes():
    """SHA-256 of every function's source, to detect edits made after the commit."""
    per = {}
    for f in CODE_FUNCS:
        try:
            per[f.__name__] = hashlib.sha256(inspect.getsource(f).encode("utf-8")).hexdigest()
        except (OSError, TypeError):
            per[f.__name__] = None
    return {"combined": sha256_json(per), "per_function": per}


def _write_csv(path, rows, fields):
    with open(path, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=fields)
        wr.writeheader()
        wr.writerows(rows)


def _jsonable(v):
    if isinstance(v, float) and not np.isfinite(v):
        return None
    if isinstance(v, dict):
        return {k: _jsonable(x) for k, x in v.items()}
    return v


def run_ablation():
    lbls = gt_label_paths()
    missing = [p.name for p in lbls if not cache_file(p).exists()]
    if missing:
        raise FileNotFoundError(f"{len(missing)} images are not cached (e.g. {missing[:3]}); run Step 1.")
    for name in OUTPUT_NAMES:
        if (OUT_ROOT / name / "metrics.json").exists():
            raise FileExistsError(f"Results already exist in {OUT_ROOT / name}. Start a new run.")

    acc = {name: {"TP": 0, "FP": 0, "FN": 0, "ious": [], "dices": [], "rows": [], "per_image": [],
                  "drops": [], "export_stats": [], "stage_totals": {s: 0 for s in STAGES},
                  "n_pred_empty": 0, "n_pred": 0, "zero_pred_imgs": 0} for name in OUTPUT_NAMES}
    n_gt_total, gt_lines_skipped, zero_gt_imgs = 0, 0, 0

    def accumulate(a, img_name, res, trace, drops, estats, n_gt, n_pred):
        for k in ["TP", "FP", "FN"]:
            a[k] += res[k]
        a["ious"] += res["ious"]
        a["dices"] += res["dices"]
        a["rows"] += res["rows"]
        a["n_pred_empty"] += res["n_pred_empty"]
        a["n_pred"] += n_pred
        a["zero_pred_imgs"] += int(n_pred == 0)
        a["drops"] += [{"image": img_name, **d} for d in drops]
        a["export_stats"] += [{"image": img_name, **s} for s in estats]
        for s in STAGES:
            a["stage_totals"][s] += len(trace[s])
        a["per_image"].append({"image": img_name, "n_gt": n_gt, "n_pred": n_pred,
                               "n_dropped_at_export": len(drops),
                               "TP": res["TP"], "FP": res["FP"], "FN": res["FN"],
                               **{f"n_after_{s}": len(trace[s]) for s in STAGES}})

    for lbl in tqdm(lbls, desc="Ablation B0-B3"):
        entry = load_cache_entry(lbl)
        h, w, img_name = entry["h"], entry["w"], entry["image_name"]
        gt_masks = read_yolo_seg_masks(lbl, w, h)
        n_lines = sum(1 for ln in lbl.read_text().splitlines() if ln.strip())
        gt_lines_skipped += n_lines - len(gt_masks)
        gts = list(enumerate(gt_masks))            # gt_id = polygon index in the GT file
        n_gt_total += len(gts)
        zero_gt_imgs += int(len(gts) == 0)

        for name, flags in CONFIGS.items():
            final, trace = run_config(entry, flags)
            label_path = OUT_ROOT / name / "labels" / f"{Path(img_name).stem}.txt"
            preds, kept, drops, estats = export_instances(final, h, w, label_path)
            res = evaluate_image(img_name, preds, gts)
            accumulate(acc[name], img_name, res, trace, drops, estats, len(gts), len(preds))
            if name == "B3":
                # Same instances as exported B3, but scored on their raw SAM masks
                raw_preds = [(d["id"], d["mask"]) for d in kept]
                res_raw = evaluate_image(img_name, raw_preds, gts)
                accumulate(acc[RAWMASK_NAME], img_name, res_raw, trace, drops, [], len(gts), len(raw_preds))

    # ---------------- provenance shared by all configs ----------------
    cache_sha = sha256_json({p.name: sha256_file(cache_file(p)) for p in lbls})
    cfg_files_sha = {p.name: sha256_file(p) for p in [PIPELINE_CFG, SEED_INPUT] if p.exists()}
    code_sha = code_hashes()
    created = datetime.now().isoformat(timespec="seconds")

    summary_rows = []
    for name in OUTPUT_NAMES:
        a, d = acc[name], OUT_ROOT / name
        d.mkdir(parents=True, exist_ok=True)
        snap = config_snapshot(name)
        (d / "config_snapshot.json").write_text(json.dumps(snap, indent=2))
        m = summarise(a["TP"], a["FP"], a["FN"], a["ious"], a["dices"])
        es = a["export_stats"]
        metrics = {
            "config": name, **m,
            "n_images": len(lbls), "is_full_run": IS_FULL_RUN,
            "n_gt_instances": n_gt_total, "n_pred_instances": a["n_pred"],
            "n_dropped_at_export": len(a["drops"]),
            "n_images_zero_gt": zero_gt_imgs, "n_images_zero_pred": a["zero_pred_imgs"],
            "n_pred_empty_after_rasterise": a["n_pred_empty"],
            "gt_lines_skipped_by_reader": gt_lines_skipped,
            "instances_after_each_stage": a["stage_totals"],
            "export_loss": None if name == RAWMASK_NAME else {
                "n_exported": len(es),
                "n_multi_contour_masks": sum(1 for s in es if s["n_outer_contours"] > 1),
                "mean_iou_raw_vs_exported": float(np.mean([s["iou_raw_vs_exported"] for s in es])) if es else None,
                "total_area_retained": (sum(s["exported_area_px"] for s in es) /
                                        max(sum(s["raw_area_px"] for s in es), 1)) if es else None,
            },
            "provenance": {
                "git_commit": GIT_COMMIT, "notebook_version": NOTEBOOK_VERSION,
                "config_snapshot_sha256": sha256_json(snap),
                "code_sha256": code_sha["combined"], "code_sha256_per_function": code_sha["per_function"],
                "config_files_sha256": cfg_files_sha, "cache_sha256": cache_sha,
                "run_tag": RUN_TAG, "created": created, "environment": "colab" if IN_COLAB else "local",
                "device": device, "python": platform.python_version(), "torch": torch.__version__,
                "transformers": transformers.__version__, "seed": SEED,
            },
        }
        (d / "metrics.json").write_text(json.dumps(_jsonable(metrics), indent=2))
        _write_csv(d / "per_instance.csv", a["rows"], PER_INSTANCE_FIELDS)
        _write_csv(d / "per_image.csv", a["per_image"],
                   ["image", "n_gt", "n_pred", "n_dropped_at_export", "TP", "FP", "FN"]
                   + [f"n_after_{s}" for s in STAGES])
        _write_csv(d / "export_drops.csv", a["drops"], ["image", "pred_id", "raw_area_px", "reason"])
        if name != RAWMASK_NAME:
            _write_csv(d / "export_stats.csv", es, ["image", "pred_id", "n_outer_contours", "raw_area_px",
                                                    "exported_area_px", "iou_raw_vs_exported"])
        fmt = lambda v: "" if v is None else round(v, 4)
        summary_rows.append({"Config": name, "TP": m["TP"], "FP": m["FP"], "FN": m["FN"],
                             "Precision": fmt(m["precision"]), "Recall": fmt(m["recall"]), "F1": fmt(m["f1"]),
                             "mean IoU": fmt(m["iou_mean"]), "mean Dice": fmt(m["dice_mean"]),
                             "dropped at export": len(a["drops"])})

    _write_csv(OUT_ROOT / "summary.csv", summary_rows, list(summary_rows[0].keys()))
    for p in [PIPELINE_CFG, SEED_INPUT]:
        if p.exists():
            shutil.copy2(p, OUT_ROOT / p.name)

    print(f"\n{len(lbls)} images | {n_gt_total} GT instances | git {GIT_COMMIT}")
    print(f"{'Config':11s} {'TP':>4s} {'FP':>4s} {'FN':>4s} {'P':>7s} {'R':>7s} {'F1':>7s} {'IoU':>7s} {'Dice':>7s} {'drop':>5s}")
    for r in summary_rows:
        print(f"{r['Config']:11s} {r['TP']:4d} {r['FP']:4d} {r['FN']:4d} {str(r['Precision']):>7s} "
              f"{str(r['Recall']):>7s} {str(r['F1']):>7s} {str(r['mean IoU']):>7s} {str(r['mean Dice']):>7s} "
              f"{r['dropped at export']:5d}")
    print("\nSaved to", OUT_ROOT)
    return summary_rows


if RUN_ABLATION:
    ABLATION_SUMMARY = run_ablation()
else:
    print("RUN_ABLATION is False; skipping.")

## Cell 14 · Step 4 (Phase 6): B0 vs B3 figure with GT overlay
One row per image: **GT | B0 | B3**. GT outlines are green on every panel; predictions are filled red. It reads the exported label files, so it shows exactly what was evaluated.

In [ ]:
# ============================================================
# Cell 14 — Step 4: comparison figure (run in Phase 6)
# ============================================================
def make_comparison_figure(image_names, run_dir=None, configs=("B0", "B3"), out_name="fig_B0_vs_B3.png"):
    run_dir = Path(run_dir) if run_dir else OUT_ROOT
    by_image = {}
    for lbl in all_gt_label_paths():
        p = label_to_image_path(lbl, EVAL_IMG_DIR)
        if p is not None:
            by_image[p.name] = (lbl, p)
    rows = []
    for name in image_names:
        lbl, img_path = by_image[name]
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        gt = read_yolo_seg_masks(lbl, w, h)

        def panel(pred_masks, title):
            out = img.copy()
            for m in pred_masks:
                sel = m > 0
                out[sel] = (0.55 * out[sel] + 0.45 * np.array([0, 0, 255])).astype(np.uint8)
                cs, _ = cv2.findContours(sel.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(out, cs, -1, (0, 0, 255), 1)
            for m in gt:
                cs, _ = cv2.findContours((m > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                cv2.drawContours(out, cs, -1, (0, 255, 0), 2)
            cv2.rectangle(out, (0, 0), (w, 34), (0, 0, 0), -1)
            cv2.putText(out, title, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
            return out

        panels = [panel([], f"GT ({len(gt)})  {name}")]
        for c in configs:
            pm = read_yolo_seg_masks(run_dir / c / "labels" / f"{Path(name).stem}.txt", w, h)
            panels.append(panel(pm, f"{c}: {len(pm)} predicted"))
        rows.append(cv2.hconcat(panels))
    fig = cv2.vconcat(rows)
    out_path = run_dir / "figures" / out_name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), fig)
    print("Figure saved:", out_path)
    return out_path


if RUN_FIGURE:
    make_comparison_figure(FIGURE_IMAGES, run_dir=FIGURE_RUN_DIR)
else:
    print("RUN_FIGURE is False; skipping (used in Phase 6).")